# 火場避難 7 類 YOLOv7-tiny：訓練 → 重參數化 → 打包轉檔

類別順序（**不能改**）：`0 cabinet 1 boxes 2 furniture 3 rubble 4 gate 5 smoke 6 person`，0–5 和舊模型相同。

開始前：
1. 上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**。
2. Google Drive 的 `我的雲端硬碟/fire_retrain/` 裡要有這 7 個檔案（就是本機 `Documents\fire_retrain_data\upload_to_drive\` 整個資料夾的內容）：
   `fire7.zip`、`fire7_quant_images.zip`、`best.pt`、`yolov7-tiny-fire7.yaml`、`yolov7-tiny-fire7-deploy.yaml`、`hyp.ft7.yaml`、`reparam.py`
3. 從上到下一格一格執行（Shift+Enter）。訓練結果會直接存在 Drive，斷線後可以接續。

## 1. 確認 GPU
應該看到 `Tesla T4`（或其他 GPU）。如果顯示 `NVIDIA-SMI has failed`，表示沒選 GPU，回去改執行階段類型。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 掛載 Google Drive，檢查檔案
會跳出授權視窗，選自己的帳號並允許。下面會逐一檢查 7 個檔案是否都在；缺檔時會停下來並列出缺哪些。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
D = '/content/drive/MyDrive/fire_retrain'   # Drive 上放檔案的資料夾
NEED = ['fire7.zip', 'fire7_quant_images.zip', 'best.pt', 'yolov7-tiny-fire7.yaml',
        'yolov7-tiny-fire7-deploy.yaml', 'hyp.ft7.yaml', 'reparam.py']
missing = [f for f in NEED if not os.path.exists(f'{D}/{f}')]
for f in NEED:
    if f not in missing:
        print(f'OK  {f:32s} {os.path.getsize(f"{D}/{f}") / 1e6:8.1f} MB')
if missing:
    raise SystemExit(f'Drive 的 {D} 缺少：{missing}')

## 3. 下載 YOLOv7，修正和新版 PyTorch／NumPy 不相容的地方
- 不執行 `pip install -r requirements.txt`：它要求 `numpy<1.24`，和 Colab 的 Python 版本衝突。Colab 已經內建需要的套件，這裡只補裝 `thop`。
- 自動修補 4 個已知問題，每項都會印出改了幾處（印 0 處表示 repo 已經修好，不影響）：
  1. `np.int` / `np.float` 在新版 NumPy 已移除
  2. OTA loss 的 `zeros_like(..., device="cpu")` 在 GPU 上會出現 `indices should be either on cpu or on the same device`
  3. `gj.clamp_(0, gain[3] - 1)`：long 和 float 混用，出現 `result type Float can't be cast to ... long`
  4. PyTorch ≥ 2.6 的 `torch.load` 預設只讀權重，用環境變數關掉

In [ ]:
%cd /content
!rm -rf yolov7 && git clone -q --depth 1 https://github.com/WongKinYiu/yolov7
%cd /content/yolov7
!pip -q install thop

import re, glob
def patch(pattern, repl, files, desc):
    n = 0
    for p in files:
        s = open(p, encoding='utf-8').read()
        s2, k = re.subn(pattern, repl, s)
        if k:
            open(p, 'w', encoding='utf-8').write(s2)
            n += k
    print(f'{desc}: 改了 {n} 處')

py = glob.glob('*.py') + glob.glob('utils/*.py') + glob.glob('models/*.py')
patch(r'np\.int\b', 'int', py, 'np.int → int')
patch(r'np\.float\b', 'float', py, 'np.float → float')
patch(r'torch\.zeros_like\(cost, device="cpu"\)', 'torch.zeros_like(cost)', ['utils/loss.py'], 'OTA loss 裝置')
patch(r'gj\.clamp_\(0, gain\[3\] - 1\), gi\.clamp_\(0, gain\[2\] - 1\)',
      'gj.clamp_(0, gain[3].long() - 1), gi.clamp_(0, gain[2].long() - 1)', ['utils/loss.py'], 'clamp 型別')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'   # 之後所有 !python 指令都會繼承

!mkdir -p cfg/fire7
!cp {D}/yolov7-tiny-fire7.yaml {D}/yolov7-tiny-fire7-deploy.yaml {D}/hyp.ft7.yaml cfg/fire7/
!cp {D}/reparam.py .
!ls cfg/fire7

## 4. 解壓資料集
解壓到 Colab 本機硬碟（`/content/fire7`），讀圖比直接讀 Drive 快很多。`data.yaml` 裡的路徑已經寫好 `/content/fire7`。

In [ ]:
!rm -rf /content/fire7 && unzip -q {D}/fire7.zip -d /content/
!cat /content/fire7/data.yaml
for sp in ('train', 'val'):
    imgs = len(os.listdir(f'/content/fire7/images/{sp}'))
    labs = os.listdir(f'/content/fire7/labels/{sp}')
    empty = sum(os.path.getsize(f'/content/fire7/labels/{sp}/{f}') == 0 for f in labs)
    print(f'{sp}: {imgs} 張圖、{len(labs)} 個標註檔（其中 {empty} 個是空的負樣本）')

## 5. 訓練（從 6 類的 best.pt 微調）
- `--weights best.pt`：nc 從 6 變 7，偵測頭形狀不同，yolov7 只會載入形狀相同的層。log 開頭會印 `Transferred xxx/yyy items`；backbone 會保留，偵測頭重新學。
- `--hyp hyp.ft7.yaml`：lr0 0.006、warmup 2、hsv_v 0.5、mixup 0.1（其他和舊模型的 hyp.ft.yaml 相同）。
- `--img 576 576`：上板時是 576×320，推論時會 letterbox，兩者相容。
- **建議 `EPOCHS = 80`**：約 3000 張訓練圖、batch 32，每輪約 95 步。偵測頭要從頭學，所以比舊模型續訓的 36 輪多；預計 T4 上約 1.5–2.5 小時。可以先設 5 跑通，確認沒問題再改回 80。
- `--project` 指到 Drive：每輪的 `last.pt`、`best.pt` 都直接存在 Drive。斷線後，把最下面那行 `RESUME = True` 再執行這格，就會從 `last.pt` 接續。

In [ ]:
EPOCHS = 80
BATCH = 32          # 出現 CUDA out of memory 就改 16
RESUME = False
RUN = f'{D}/runs/fire7'

if RESUME:
    !python train.py --resume {RUN}/weights/last.pt
else:
    !python train.py --device 0 --workers 2 --batch-size {BATCH} \
      --data /content/fire7/data.yaml --img 576 576 \
      --cfg cfg/fire7/yolov7-tiny-fire7.yaml --hyp cfg/fire7/hyp.ft7.yaml \
      --weights {D}/best.pt --epochs {EPOCHS} \
      --project {D}/runs --name fire7 --exist-ok --cache-images

## 6. 訓練曲線、PR 曲線、混淆矩陣
- `results.png`：每輪的 loss、precision、recall、mAP。
- `PR_curve.png`：每類的 precision-recall 曲線，圖例裡有各類 mAP@0.5。
- `confusion_matrix.png`：最後一列／欄的 `background` 是「把背景當成物件（誤判）」和「漏抓」，這次要特別看 `background` 那一列，數字越小代表誤判越少。

這格用 `best.pt` 重新在 val 上測一次，確保 PR 曲線和混淆矩陣是最佳權重的結果。

In [ ]:
!python test.py --data /content/fire7/data.yaml --img 576 --batch 32 --conf 0.001 --iou 0.65 \
  --weights {RUN}/weights/best.pt --task val --project {D}/runs --name fire7_val --exist-ok

from IPython.display import Image, display
for p in [f'{RUN}/results.png', f'{D}/runs/fire7_val/PR_curve.png', f'{D}/runs/fire7_val/confusion_matrix.png']:
    if os.path.exists(p):
        print(p); display(Image(filename=p, width=900))
    else:
        print('找不到', p)

## 7. 負樣本誤判檢查
只看 val 裡的負樣本（空場景渲染 `empty_`、教室照片 `neg_`），用上板的門檻 0.4 偵測，數出有多少張出現框。
舊模型在教室實景「每張都有一堆框，最高 89%」，新模型這個比例要明顯降低才算有效。

In [ ]:
!rm -rf /content/fp && python detect.py --weights {RUN}/weights/best.pt --source /content/fire7/images/val \
  --img 576 --conf 0.4 --save-txt --save-conf --nosave --project /content/fp --name val --exist-ok > /dev/null

val_imgs = os.listdir('/content/fire7/images/val')
neg = [f for f in val_imgs if os.path.getsize(f'/content/fire7/labels/val/{os.path.splitext(f)[0]}.txt') == 0]
hit = [f for f in neg if os.path.exists(f'/content/fp/val/labels/{os.path.splitext(f)[0]}.txt')]
print(f'val 負樣本 {len(neg)} 張，門檻 0.4 下出現框的有 {len(hit)} 張（{len(hit) / max(1, len(neg)):.1%}）')
for f in hit[:10]:
    print('  ', f, open(f'/content/fp/val/labels/{os.path.splitext(f)[0]}.txt').read().strip().replace(chr(10), ' | '))

## 8. 重參數化（IDetect → Detect）
`reparam.py` 會把訓練版最後一層的隱式參數合併進卷積：`W' = im·W`、`b' = im·(b + Σ W·ia)`。
它會用 1×3×320×576 的隨機輸入比對兩個模型，最大差異 < 1e-3 才存檔，所以看到 `已存 best_reparam.pt` 就代表合併正確。
接著再用部署版跑一次 val，mAP 應該和第 6 格相同。

In [ ]:
!python reparam.py --weights {RUN}/weights/best.pt --cfg cfg/fire7/yolov7-tiny-fire7-deploy.yaml --out /content/best_reparam.pt
!python test.py --data /content/fire7/data.yaml --img 576 --batch 32 --conf 0.001 --iou 0.65 \
  --weights /content/best_reparam.pt --task val --project /content/runs_test --name reparam --exist-ok 2>&1 | grep -E "all|Class"

## 9. 打包 Realtek 線上轉檔要的 zip
產生兩個檔案，同時存到 Drive 的 `fire_retrain/convert_upload/`，並直接下載到電腦：
- `best_reparam_pt.zip`：裡面只有 `best_reparam.pt`（上傳上限約 35 MB，tiny 約 12 MB）
- `fire7_quant_images.zip`：30 張 576×320 的量化校正圖（本機已經挑好，這裡只是一起放進去）

轉檔網頁要填的參數：**YOLOv7-TINY-Pytorch、輸入 576 × 320、UINT8、類別數 7**。

In [ ]:
OUT = f'{D}/convert_upload'
!mkdir -p {OUT}
!cd /content && rm -f best_reparam_pt.zip && zip -j best_reparam_pt.zip best_reparam.pt
!cp /content/best_reparam_pt.zip {D}/fire7_quant_images.zip {OUT}/
!ls -la {OUT}
!unzip -l {OUT}/best_reparam_pt.zip

from google.colab import files
files.download('/content/best_reparam_pt.zip')
files.download(f'{OUT}/fire7_quant_images.zip')